# Overview

This notebook is designed for the CatBoost model training.
We will use gradient boosting with extracted images embeddings in order to predict the litotypes on the source images.

Unfortunately, CatBoost doesn't natively support MPS, so CPU calculations will be used instead. 

## 1. Imports and Settings

In [ ]:
import os
import ast
import datetime
import warnings

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold

from pyprojroot import here
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [2]:
BASE_PATH = here()
TARGET_MODEL_TYPE = "gb"
BACKEND_NAME = "CatBoost"

DATASET_PATH = "data/meta/processed/"
DATASET_TARGET_FILE = "metadata_dinov3_embeddings.parquet"

FEATURE_COLS = [
    "interval_start", "interval_end"
]
SHOULD_INCLUDE_LBA_EMBEDDINGS = False
SHOULD_INCLUDE_SLUDGE_EMBEDDINGS = True
TARGET_COLS = [
    'sandstone_sludge', 'siltstone_sludge', 'argillite_sludge'
]

FINAL_MODEL_IGNORED_WELL_ID = 1

OUTPUT_PATH = "output/"
OUTPUT_MODELS_PATH = "models/"
OUTPUT_PREDICTIONS_PATH = "predictions/"
OUTPUT_RESULT_MODEL_NAME = "[{0}] {1}.cbm"
OUTPUT_PREDICTIONS_NAME = "[{0}] {1} ({2}).csv"

SHOULD_SAVE_FINAL_MODEL = True
SHOULD_SAVE_FINAL_PREDICTIONS = True

In [3]:
warnings.filterwarnings("ignore")

np.random.seed(42)

## 2. Dataset Loading

In [ ]:
df = pd.read_parquet(os.path.join(BASE_PATH, DATASET_PATH, TARGET_MODEL_TYPE, DATASET_TARGET_FILE))

print("Dataset loaded. Sample: ")
print(df.head())

Dataset loaded. Sample: 
   well_id  device_no  interval_start  interval_end  sandstone_sludge  \
0        4        1.0            2675          2680                 5   
1        4        2.0            2680          2685                 5   
2        4        3.0            2685          2690                 5   
3        4        4.0            2690          2695                 5   
4        4        5.0            2695          2700                 5   

   siltstone_sludge  argillite_sludge  radiolarite_sludge  coal_sludge  \
0                25                70                   0            0   
1                25                70                   0            0   
2                30                65                   0            0   
3                30                65                   0            0   
4                25                70                   0            0   

   limestone_sludge  ...  oil_saturation  calcite_carbonatometry  \
0                 0  ..

## 3. Embeddings Processing

In [5]:
def str_to_array(x):
    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array(x)
    return x

df['lba_dinov3_emb'] = df['lba_dinov3_emb'].apply(str_to_array)
df['sludge_dinov3_emb'] = df['sludge_dinov3_emb'].apply(str_to_array)

print("Embeddings formatted as arrays.")
print(df['sludge_dinov3_emb'].values[0].shape)

Embeddings formatted as arrays.
(1024,)


## 4. Features and Targets Preparation

In [6]:
# We reduce 1024-dimensional embeddings to 32 principal components
N_PCA_COMPONENTS = 32

pca_lba = PCA(n_components=N_PCA_COMPONENTS, random_state=42)
lba_emb_pca = pca_lba.fit_transform(df["lba_dinov3_emb"].tolist())

pca_sludge = PCA(n_components=N_PCA_COMPONENTS, random_state=42)
sludge_emb_pca = pca_sludge.fit_transform(df["sludge_dinov3_emb"].tolist())

lba_emb_df = pd.DataFrame(
    lba_emb_pca, columns=[f"lba_emb_pca_{i}" for i in range(N_PCA_COMPONENTS)]
)
sludge_emb_df = pd.DataFrame(
    sludge_emb_pca, columns=[f"sludge_emb_pca_{i}" for i in range(N_PCA_COMPONENTS)]
)

input_df = pd.concat(
    [
        df[FEATURE_COLS].reset_index(drop=True),
        lba_emb_df if SHOULD_INCLUDE_LBA_EMBEDDINGS else None,
        sludge_emb_df if SHOULD_INCLUDE_SLUDGE_EMBEDDINGS else None,
    ],
    axis=1,
)
print("Input features are prepared (with PCA).")
print(input_df.head())

output_df = df[TARGET_COLS].reset_index(drop=True)
print("Target variables are prepared.")
print(output_df.head())


Input features are prepared (with PCA).
   interval_start  interval_end  sludge_emb_pca_0  sludge_emb_pca_1  \
0            2675          2680          6.624063          5.625460   
1            2680          2685          5.128608          4.668198   
2            2685          2690          5.303943          5.179303   
3            2690          2695          5.428961          1.753567   
4            2695          2700          5.147763          2.543802   

   sludge_emb_pca_2  sludge_emb_pca_3  sludge_emb_pca_4  sludge_emb_pca_5  \
0         -1.882981          0.600944         -0.580181         -1.466452   
1         -1.340740          2.326202          1.439810         -1.539011   
2         -1.252371          1.704347          1.450878         -1.591405   
3         -0.735190          2.218346          3.106397         -1.658281   
4         -0.827202          3.639742          2.980572         -0.300330   

   sludge_emb_pca_6  sludge_emb_pca_7  ...  sludge_emb_pca_22  \
0    

## 5. Training

### 5.1. Hyper-Params

In [7]:
catboost_params = {
    'iterations': 5000,
    'learning_rate': 0.015,
    'depth': 5,
    'l2_leaf_reg': 10,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'rsm': 0.7,
    'loss_function': 'MultiRMSE',
    'eval_metric': 'MultiRMSE',
    'random_seed': 42,
    'early_stopping_rounds': 200,
    'verbose': 100,
    'task_type': 'CPU',
    'devices': '0'
}

### 5.2. Helpers Declaration

In [8]:
class TabularDataset:
    def __init__(self, X, y, groups = None):
        self.X = X
        self.y = y
        self.groups = groups

    def get_fold(self, train_idx, val_idx):
        X_train = self.X.iloc[train_idx]
        X_val = self.X.iloc[val_idx]

        y_train = self.y.iloc[train_idx]
        y_val = self.y.iloc[val_idx]

        return X_train, X_val, y_train, y_val

    def extract_folds(self, n_splits = 4):
        gkf = GroupKFold(n_splits = n_splits)
        folds = []
        for train_idx, val_idx in gkf.split(self.X, self.y, self.groups):
            X_train = self.X.iloc[train_idx]
            X_val = self.X.iloc[val_idx]
            y_train = self.y.iloc[train_idx]
            y_val = self.y.iloc[val_idx]
            folds.append((X_train, X_val, y_train, y_val))
        return folds


In [9]:
def normalize_predictions(predictions):
    clipped = np.clip(predictions, 0, None)
    predictions_sum = clipped.sum(axis = 1, keepdims = True)
    predictions_sum = np.where(predictions_sum == 0, 1.0, predictions_sum)
    return clipped / predictions_sum * 100

def train_model(X_train, y_train, X_val, y_val, params):
    model = CatBoostRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set = (X_val, y_val),
        use_best_model = True,
        verbose = True
    )
    return model

def train_model_no_eval(X_train, y_train, params):
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, verbose = False)
    return model

def save_model_to_disk(model, filepath):
    model.save_model(filepath)

def save_predictions_to_csv(predictions, y_true, filepath):
    pred_renamed = predictions.rename(columns = lambda x: f"{x}_pred")
    y_true_renamed = y_true.rename(columns = lambda x: f"{x}_true")
    combined_df = pd.concat([pred_renamed, y_true_renamed], axis = 1)
    combined_df.to_csv(filepath)

def get_predictions_by_model(model, X_val, y_val):
    pred = model.predict(X_val)
    pred = normalize_predictions(pred)
    pred_df = pd.DataFrame(
        pred,
        columns = y_val.columns,
        index = y_val.index
    )
    return pred_df

def calculate_metrics(
    y_true: pd.DataFrame,
    y_predicted: pd.DataFrame
) -> dict:
    metrics = {}
    for target in y_true.columns:
        metrics[target] = {
            "mae": mean_absolute_error(
                y_true[target],
                y_predicted[target]
            ),
            "rmse": root_mean_squared_error(
                y_true[target],
                y_predicted[target]
            ),
            "r2": r2_score(
                y_true[target],
                y_predicted[target]
            )
        }
    return metrics

def print_metrics(metrics):
    print("!== Cross-Validation Metrics ==!")
    for target in metrics:
        print(target)
        # MAE:
        print(
            f"MAE: {np.mean(metrics[target]['mae']):.4f} "
            f"+/- {np.std(metrics[target]['mae']):.4f}"
        )
        # RMSE:
        print(
            f"RMSE: {np.mean(metrics[target]['rmse']):.4f} "
            f"+/- {np.std(metrics[target]['rmse']):.4f}"
        )
        # R2:
        print(
            f"R2: {np.mean(metrics[target]['r2']):.4f} "
            f"+/- {np.std(metrics[target]['r2']):.4f}"
        )
        print("===")


### 5.3. Orchestration Functions

In [10]:
def run_cv(
    input_df,
    output_df,
    groups,
    params,
    n_splits = 4
):
    dataset = TabularDataset(input_df, output_df, groups)
    folds = dataset.extract_folds(n_splits = n_splits)
    
    aggregated_metrics = {
        target: {
            "mae": [],
            "rmse": [],
            "r2": []
        }
        for target in output_df.columns
    }
    
    # Ensure directory for metrics exists
    os.makedirs(os.path.join(BASE_PATH, OUTPUT_PATH, TARGET_MODEL_TYPE, OUTPUT_PREDICTIONS_PATH), exist_ok = True)
    
    for fold_idx, (X_train, X_val, y_train, y_val) in enumerate(folds):
        model = train_model(X_train, y_train, X_val, y_val, params)
        predictions = get_predictions_by_model(model, X_val, y_val)
        
        current_datetime = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
        filename = OUTPUT_PREDICTIONS_NAME.format(current_datetime, BACKEND_NAME, f"Fold {fold_idx + 1}")
        filepath = os.path.join(BASE_PATH, OUTPUT_PATH, TARGET_MODEL_TYPE, OUTPUT_PREDICTIONS_PATH, filename)
        save_predictions_to_csv(predictions, y_val, filepath)
        
        fold_metrics = calculate_metrics(y_val, predictions)
        
        for target in fold_metrics:
            for metric_name in fold_metrics[target]:
                aggregated_metrics[target][metric_name].append(
                    fold_metrics[target][metric_name]
                )
                
        print(f"Fold {fold_idx + 1} completed.")
        
    return aggregated_metrics

def run_final_training(
    input_df,
    output_df,
    groups,
    params,
    should_save_model = True,
    should_save_predictions = True
):
    train_mask = groups != FINAL_MODEL_IGNORED_WELL_ID
    test_mask = groups == FINAL_MODEL_IGNORED_WELL_ID
    current_datetime = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

    X_train = input_df[train_mask]
    y_train = output_df[train_mask]
    X_test = input_df[test_mask]
    y_test = output_df[test_mask]

    print(f"Training final model on wells: {groups[train_mask].unique()}.")
    print(f"Testing final model on holdout well: {FINAL_MODEL_IGNORED_WELL_ID} ({len(X_test)} samples).")

    model = train_model_no_eval(X_train, y_train, params)
    test_predictions = get_predictions_by_model(model, X_test, y_test)

    if should_save_model:
        model_name = OUTPUT_RESULT_MODEL_NAME.format(current_datetime, BACKEND_NAME)
        model_output_path = os.path.join(BASE_PATH, OUTPUT_PATH, TARGET_MODEL_TYPE, OUTPUT_MODELS_PATH, model_name)
        save_model_to_disk(model, model_output_path)
        print(f"Holdout model saved to: {model_output_path}.")


    if should_save_predictions:
        test_filename = OUTPUT_PREDICTIONS_NAME.format(current_datetime, BACKEND_NAME, "Result")
        test_filepath = os.path.join(BASE_PATH, OUTPUT_PATH, TARGET_MODEL_TYPE, OUTPUT_PREDICTIONS_PATH, test_filename)
        save_predictions_to_csv(test_predictions, y_test, test_filepath)
        print(f"Holdout predictions saved to: {test_filepath}.")

    test_metrics = calculate_metrics(y_test, test_predictions)
    return model, test_metrics

### 5.4. Training / Saving

In [11]:
groups = df["well_id"]
metrics = run_cv(
    input_df,
    output_df,
    groups,
    catboost_params,
    n_splits = 4
)
print_metrics(metrics)

0:	learn: 39.6806429	test: 77.6056840	best: 77.6056840 (0)	total: 59.9ms	remaining: 4m 59s
1:	learn: 39.2773337	test: 77.7149057	best: 77.6056840 (0)	total: 62.3ms	remaining: 2m 35s
2:	learn: 38.9033337	test: 77.8587266	best: 77.6056840 (0)	total: 64.2ms	remaining: 1m 46s
3:	learn: 38.5151773	test: 77.9981124	best: 77.6056840 (0)	total: 65.7ms	remaining: 1m 22s
4:	learn: 38.1528294	test: 78.1187958	best: 77.6056840 (0)	total: 66.7ms	remaining: 1m 6s
5:	learn: 37.7690223	test: 78.2830780	best: 77.6056840 (0)	total: 67.9ms	remaining: 56.6s
6:	learn: 37.4446635	test: 78.5257145	best: 77.6056840 (0)	total: 69.1ms	remaining: 49.3s
7:	learn: 37.0891505	test: 78.6197361	best: 77.6056840 (0)	total: 70.4ms	remaining: 44s
8:	learn: 36.7094176	test: 78.7254418	best: 77.6056840 (0)	total: 72.1ms	remaining: 40s
9:	learn: 36.3781404	test: 78.8560372	best: 77.6056840 (0)	total: 74ms	remaining: 36.9s
10:	learn: 36.0497448	test: 78.9858833	best: 77.6056840 (0)	total: 75.2ms	remaining: 34.1s
11:	learn: 

In [12]:
groups = df["well_id"]
model, test_metrics = run_final_training(
    input_df,
    output_df,
    groups,
    catboost_params,
    should_save_model = SHOULD_SAVE_FINAL_MODEL,
    should_save_predictions = SHOULD_SAVE_FINAL_PREDICTIONS
)
print_metrics(test_metrics)

print("Final Features:")
print(model.get_feature_importance(prettified = True)) # type: ignore

Training final model on wells: [4 2 3].
Testing final model on holdout well: 1 (390 samples).
Holdout model saved to: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/output/gb/models/[2026-06-23T13:35:46] CatBoost.cbm.
Holdout predictions saved to: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/output/gb/predictions/[2026-06-23T13:35:46] CatBoost (Result).csv.
!== Cross-Validation Metrics ==!
sandstone_sludge
MAE: 11.2703 +/- 0.0000
RMSE: 14.4411 +/- 0.0000
R2: 0.8341 +/- 0.0000
===
siltstone_sludge
MAE: 5.0861 +/- 0.0000
RMSE: 6.6427 +/- 0.0000
R2: 0.7472 +/- 0.0000
===
argillite_sludge
MAE: 11.7142 +/- 0.0000
RMSE: 13.3629 +/- 0.0000
R2: 0.6855 +/- 0.0000
===
Final Features:
           Feature Id  Importances
0    sludge_emb_pca_2    21.873494
1    sludge_emb_pca_0    21.600053
2      interval_start    18.842034
3    sludge_emb_pca_1    13.3